<a href="https://colab.research.google.com/github/Dartrint/BioMistral_chatbot/blob/main/BioMistral_chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Build BioMistral Medical RAG Chatbot using BioMistral Open Source LLM

In the notebook we will build a Medical Chatbot with BioMistral LLM and Heart Health pdf file.

## Installation

In [2]:
!pip uninstall -y langchain langchain-core langchain-community pydantic \
                llama-cpp-python chromadb pypdf sentence-transformers

!pip install \
  langchain==0.2.14 \
  langchain-core==0.2.33 \
  langchain-community==0.2.12 \
  pydantic==2.7.4 \
  llama-cpp-python \
  chromadb \
  pypdf \
  sentence-transformers \
  faiss-cpu

  Using cached langchain-0.2.14-py3-none-any.whl.metadata (7.1 kB)
  Using cached langchain_core-0.2.33-py3-none-any.whl.metadata (6.2 kB)
  Using cached langchain_community-0.2.12-py3-none-any.whl.metadata (2.7 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 8.4 MB/s eta 0:00:00
  Using cached llama_cpp_python-0.3.20-py3-none-linux_x86_64.whl
  Using cached chromadb-1.5.8-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (5.0 kB)
  Using cached pypdf-6.10.2-py3-none-any.whl.metadata (7.1 kB)
  Using cached sentence_transformers-5.4.1-py3-none-any.whl.metadata (17 kB)
  Using cached faiss_cpu-1.13.2-cp310-abi3-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (7.6 kB)
  Using cached langchain_text_splitters-0.2.4-py3-none-any.whl.metadata (2.3 kB)
  Using cached langsmith-0.1.147-py3-none-any.whl.metadata (14 kB)
  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached tenacity-8.5.0-py

## Import libraries

In [1]:
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import CharacterTextSplitter,RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS, Chroma
from langchain_community.llms import LlamaCpp
# RetrievalQA and LLMChain are not used in the subsequent LCEL RAG chain, so they can be removed.
# from langchain_community.chains import RetrievalQA, LLMChain

In [2]:
import pathlib
import textwrap
from IPython.display import display
from IPython.display import Markdown



def to_markdown(text):
  text = text.replace('•', '  *')
  return Markdown(textwrap.indent(text, '> ', predicate=lambda _: True))

In [3]:
# Used to securely store your API key
from google.colab import userdata

## Setup HuggingFace Access Token

- Log in to [HuggingFace.co](https://huggingface.co/)
- Click on your profile icon at the top-right corner, then choose [“Settings.”](https://huggingface.co/settings/)
- In the left sidebar, navigate to [“Access Token”](https://huggingface.co/settings/tokens)
- Generate a new access token, assigning it the “write” role.


In [4]:
# Or use `os.getenv('HUGGINGFACEHUB_API_TOKEN')` to fetch an environment variable.
import os
from getpass import getpass

HUGGINGFACEHUB_API_TOKEN = userdata.get("BioMistral_chatbot")
os.environ["BioMistral_chatbot"] = "BioMistral_chatbot"

## Import document

In [5]:
loader = PyPDFDirectoryLoader("/content/sample_data/Data")
docs = loader.load()

In [6]:
docs

[Document(metadata={'source': '/content/sample_data/Data/healthyheart.pdf', 'page': 0}, page_content='YOUR GUIDE TO\nA Healthy Heart\nU.S. DEPARTMENT OF HEALTH AND HUMAN SERVICES\nNational Institutes of Health\nNational Heart, Lung, and Blood Institute\n'),
 Document(metadata={'source': '/content/sample_data/Data/healthyheart.pdf', 'page': 1}, page_content='YOUR GUIDE TO\nA Healthy Heart\nU.S. DEPARTMENT OF HEALTH AND HUMAN SERVICES\nNational Institutes of Health\nNational Heart, Lung, and Blood Institute\nNIH Publication No. 06-5269\nDecember 2005'),
 Document(metadata={'source': '/content/sample_data/Data/healthyheart.pdf', 'page': 2}, page_content='U.S. DEPARTMENT OF HEALTH AND HUMAN SERVICES\nNational Institutes of Health\nNational Heart, Lung, and Blood Institute\nWritten by: Marian Sandmaier'),
 Document(metadata={'source': '/content/sample_data/Data/healthyheart.pdf', 'page': 3}, page_content='Heart Disease: Why Should You Care? . . . . . . . . . . . . . . . . . . . . . . . . . 

## Text Splitting - Chunking

In [7]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = text_splitter.split_documents(docs)

In [8]:
len(chunks)

585

In [9]:
chunks[0]

Document(metadata={'source': '/content/sample_data/Data/healthyheart.pdf', 'page': 0}, page_content='YOUR GUIDE TO\nA Healthy Heart\nU.S. DEPARTMENT OF HEALTH AND HUMAN SERVICES\nNational Institutes of Health\nNational Heart, Lung, and Blood Institute')

In [10]:
chunks[1]

Document(metadata={'source': '/content/sample_data/Data/healthyheart.pdf', 'page': 1}, page_content='YOUR GUIDE TO\nA Healthy Heart\nU.S. DEPARTMENT OF HEALTH AND HUMAN SERVICES\nNational Institutes of Health\nNational Heart, Lung, and Blood Institute\nNIH Publication No. 06-5269\nDecember 2005')

In [11]:
chunks[2]

Document(metadata={'source': '/content/sample_data/Data/healthyheart.pdf', 'page': 2}, page_content='U.S. DEPARTMENT OF HEALTH AND HUMAN SERVICES\nNational Institutes of Health\nNational Heart, Lung, and Blood Institute\nWritten by: Marian Sandmaier')

In [12]:
chunks[3]

Document(metadata={'source': '/content/sample_data/Data/healthyheart.pdf', 'page': 3}, page_content='Heart Disease: Why Should You Care? . . . . . . . . . . . . . . . . . . . . . . . . . . . . 1\nWhat You Need To Know About Heart Disease . . . . . . . . . . . . . . . . . . . . . . 3\nWhat Is Heart Disease?. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 3')

In [13]:
chunks[4]

Document(metadata={'source': '/content/sample_data/Data/healthyheart.pdf', 'page': 3}, page_content='Who Is at Risk? . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 4\nHow Risk Works . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 4')

## Embeddings

In [15]:
import warnings
warnings.filterwarnings('ignore')

!pip uninstall -y transformers sentence-transformers
!pip install transformers==4.38.2 sentence-transformers==2.7.0
from langchain_community.embeddings import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-base-en-v1.5")

Found existing installation: transformers 4.57.6
Uninstalling transformers-4.57.6:
  Successfully uninstalled transformers-4.57.6
Found existing installation: sentence-transformers 5.4.1
Uninstalling sentence-transformers-5.4.1:
  Successfully uninstalled sentence-transformers-5.4.1
  Using cached transformers-4.38.2-py3-none-any.whl.metadata (130 kB)
  Using cached sentence_transformers-2.7.0-py3-none-any.whl.metadata (11 kB)
  Using cached tokenizers-0.15.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.7 kB)
Using cached transformers-4.38.2-py3-none-any.whl (8.5 MB)
Using cached sentence_transformers-2.7.0-py3-none-any.whl (171 kB)
Using cached tokenizers-0.15.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.6 MB)
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
ERROR: pip's dependency resolver does not currently take into a

## Vector Store - FAISS or ChromaDB

In [16]:
from langchain_community.vectorstores import Chroma
vectorstore = Chroma.from_documents(chunks, embeddings)

In [17]:
vectorstore

In [18]:
query = "who is at risk of heart disease"
search = vectorstore.similarity_search(query)

In [19]:
to_markdown(search[0].page_content)

> heart disease. Most women don’t know 
> they’re at risk for heart disease. I have 
> several friends who have a lot of the 
> same risk factors that I do, but they’re 
> just not tuned in to them. They need to 
> know that, and they need to take better 
> care of themselves
> .
> ANN STIEGLER
> “
> What’s Your Risk?
> ”

## Retriever

In [20]:
retriever = vectorstore.as_retriever(
    search_kwargs={'k': 5}
)

In [21]:
retriever.invoke(query)

[Document(metadata={'page': 11, 'source': '/content/sample_data/Data/healthyheart.pdf'}, page_content='heart disease. Most women don’t know \nthey’re at risk for heart disease. I have \nseveral friends who have a lot of the \nsame risk factors that I do, but they’re \njust not tuned in to them. They need to \nknow that, and they need to take better \ncare of themselves\n.\nANN STIEGLER\n“\nWhat’s Your Risk?\n”'),
 Document(metadata={'page': 8, 'source': '/content/sample_data/Data/healthyheart.pdf'}, page_content='4\nWho Is at Risk?\nRisk factors are conditions or habits that make a person more likely\nto develop a disease. They can also increase the chances that an\nexisting disease will get worse. Important risk factors for heart dis-\nease that you can do something about are cigarette smoking, high'),
 Document(metadata={'page': 21, 'source': '/content/sample_data/Data/healthyheart.pdf'}, page_content='you should take it.\nHigh Blood Cholesterol\nHigh blood cholesterol is another maj

## Large Language Model - Open Source

In [22]:
#connect to google drive
from google.colab import drive

In [23]:
drive.mount('/content/drive')

Mounted at /content/drive


In [29]:
import os
os.path.exists("/content/drive/MyDrive/Model&Data/ggml-model-Q4_K_M.gguf")

True

In [30]:
from langchain_community.llms import LlamaCpp
llm = LlamaCpp(
    model_path= "/content/drive/MyDrive/Model&Data/ggml-model-Q4_K_M.gguf",
    temperature=0.3,
    max_tokens=2048,
    top_p=1)

llama_model_loader: loaded meta data with 21 key-value pairs and 291 tensors from /content/drive/MyDrive/Model&Data/ggml-model-Q4_K_M.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = models
llama_model_loader: - kv   2:                       llama.context_length u32              = 32768
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 4096
llama_model_loader: - kv   4:                          llama.block_count u32              = 32
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 14336
llama_model_loader: - kv   6:                 llama.rope.dimension_count u32              = 128
llama_model_loader: - kv   7:                 llama.att

## RAG Chain

In [31]:
from langchain.schema.runnable import RunnablePassthrough
from langchain.schema.output_parser import StrOutputParser
from langchain.prompts import ChatPromptTemplate

In [32]:
template = """
<|context|>
You are an AI assistant that follows instruction extremely well.
Please be truthful and give direct answers
</s>
<|user|>
{query}
</s>
 <|assistant|>
"""

In [33]:
prompt = ChatPromptTemplate.from_template(template)

In [34]:
rag_chain = (
    {"context": retriever,  "query": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [35]:
response = rag_chain.invoke("what disease affect the heart?")

llama_perf_context_print:        load time =   20967.92 ms
llama_perf_context_print: prompt eval time =   20967.61 ms /    58 tokens (  361.51 ms per token,     2.77 tokens per second)
llama_perf_context_print:        eval time =   29178.58 ms /    42 runs   (  694.73 ms per token,     1.44 tokens per second)
llama_perf_context_print:       total time =   50188.53 ms /   100 tokens
llama_perf_context_print:    graphs reused =         47


In [36]:
to_markdown(response)

> Hypertrophic cardiomyopathy, dilated cardiomyopathy, and myocarditis are some diseases that affect the heart. Is there anything else you would like to know?

In [38]:
import sys

while True:
  user_input = input(f"Input Prompt: ")
  if user_input == 'exit':
    print('Exiting')
    sys.exit()
  if user_input == '':
    continue
  result = rag_chain.invoke(user_input)
  print("Answer: ",result)

Input Prompt: What is diabetes?


Llama.generate: 41 prefix-match hit, remaining 15 prompt tokens to eval
llama_perf_context_print:        load time =   20967.92 ms
llama_perf_context_print: prompt eval time =    4529.35 ms /    15 tokens (  301.96 ms per token,     3.31 tokens per second)
llama_perf_context_print:        eval time =   42251.90 ms /    61 runs   (  692.65 ms per token,     1.44 tokens per second)
llama_perf_context_print:       total time =   46848.99 ms /    76 tokens
llama_perf_context_print:    graphs reused =         60


Answer:  Diabetes is a group of metabolic diseases in which there are high blood sugar levels over a long period. Symptoms include increased thirst and urine, increased hunger, weight loss, fatigue, blurred vision, slow healing sores, and unexplained infections .
Input Prompt: Ways to reduce this disease


Llama.generate: 41 prefix-match hit, remaining 17 prompt tokens to eval
llama_perf_context_print:        load time =   20967.92 ms
llama_perf_context_print: prompt eval time =    6817.14 ms /    17 tokens (  401.01 ms per token,     2.49 tokens per second)
llama_perf_context_print:        eval time =   41408.58 ms /    60 runs   (  690.14 ms per token,     1.45 tokens per second)
llama_perf_context_print:       total time =   48287.43 ms /    77 tokens
llama_perf_context_print:    graphs reused =         61


Answer:  There are many ways to reduce the risk of getting cancer. One way is to eat a healthy diet and exercise regularly. Another way is to get vaccinated for HPV, which is a virus that can cause certain types of cancer. It's also important to avoid smoking and limit alcohol consumption.
Input Prompt: exit
Exiting


SystemExit: 